In [ ]:
!pip install datasets

In [2]:
import numpy as np
import pandas as pd
import json
import re


Data Exploration


In [3]:
from datasets import load_dataset

In [4]:
dataset=load_dataset("lavita/MedQuAD")

In [5]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer'],
        num_rows: 47441
    })
})


In [6]:
dataset.keys()

dict_keys(['train'])

In [7]:
train_data=dataset['train']

In [8]:
len(train_data)

47441

In [9]:
train_data.column_names

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [10]:
train_data.features

{'document_id': Value('string'),
 'document_source': Value('string'),
 'document_url': Value('string'),
 'category': Value('string'),
 'umls_cui': Value('string'),
 'umls_semantic_types': Value('string'),
 'umls_semantic_group': Value('string'),
 'synonyms': Value('string'),
 'question_id': Value('string'),
 'question_focus': Value('string'),
 'question_type': Value('string'),
 'question': Value('string'),
 'answer': Value('string')}

In [11]:
train_data[0]

{'document_id': '0000559',
 'document_source': 'GHR',
 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair',
 'category': None,
 'umls_cui': 'C0343073',
 'umls_semantic_types': 'T047',
 'umls_semantic_group': 'Disorders',
 'synonyms': 'KWWH',
 'question_id': '0000559-1',
 'question_focus': 'keratoderma with woolly hair',
 'question_type': 'information',
 'question': 'What is (are) keratoderma with woolly hair ?',
 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the

In [12]:
for i in range(5):
  print(train_data[i])

{'document_id': '0000559', 'document_source': 'GHR', 'document_url': 'https://ghr.nlm.nih.gov/condition/keratoderma-with-woolly-hair', 'category': None, 'umls_cui': 'C0343073', 'umls_semantic_types': 'T047', 'umls_semantic_group': 'Disorders', 'synonyms': 'KWWH', 'question_id': '0000559-1', 'question_focus': 'keratoderma with woolly hair', 'question_type': 'information', 'question': 'What is (are) keratoderma with woolly hair ?', 'answer': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of th

In [13]:
df=train_data.to_pandas()

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47441 entries, 0 to 47440
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   document_id          47436 non-null  str  
 1   document_source      47441 non-null  str  
 2   document_url         47441 non-null  str  
 3   category             32010 non-null  str  
 4   umls_cui             31417 non-null  str  
 5   umls_semantic_types  31375 non-null  str  
 6   umls_semantic_group  31417 non-null  str  
 7   synonyms             24669 non-null  str  
 8   question_id          47441 non-null  str  
 9   question_focus       47427 non-null  str  
 10  question_type        47441 non-null  str  
 11  question             47441 non-null  str  
 12  answer               16407 non-null  str  
dtypes: str(13)
memory usage: 35.7 MB


In [15]:
df.shape

(47441, 13)

In [16]:
df.isnull().sum()

document_id                5
document_source            0
document_url               0
category               15431
umls_cui               16024
umls_semantic_types    16066
umls_semantic_group    16024
synonyms               22772
question_id                0
question_focus            14
question_type              0
question                   0
answer                 31034
dtype: int64

In [17]:
df.duplicated().sum()

np.int64(0)

In [18]:
df['question'].str.len().describe()

count    47441.000000
mean        51.537531
std         15.771672
min         14.000000
25%         40.000000
50%         51.000000
75%         62.000000
max        191.000000
Name: question, dtype: float64

In [19]:
df['answer'].str.len().describe()

count    16407.000000
mean      1303.452673
std       1656.694326
min          6.000000
25%        487.000000
50%        890.000000
75%       1589.000000
max      29046.000000
Name: answer, dtype: float64

In [20]:
df.to_csv("medquad_raw.csv",index=False)

Data Preprocessing


In [21]:
df=pd.read_csv("medquad_raw.csv")

In [22]:
df.columns.tolist()

['document_id',
 'document_source',
 'document_url',
 'category',
 'umls_cui',
 'umls_semantic_types',
 'umls_semantic_group',
 'synonyms',
 'question_id',
 'question_focus',
 'question_type',
 'question',
 'answer']

In [23]:
# remove missing question/answer
df = df.dropna(subset=["question", "answer"])

print("Shape after removing missing values:", df.shape)

Shape after removing missing values: (16407, 13)


In [24]:
# clean whitespace
df["question"] = df["question"].str.strip()
df["answer"] = df["answer"].str.strip()

In [25]:
before = len(df)

df = df.drop_duplicates(
    subset=["question", "answer"]
).reset_index(drop=True)

after = len(df)

print("Duplicates removed:", before - after)
print("Final records:", after)

Duplicates removed: 48
Final records: 16359


In [26]:
print(df.head(5))

  document_id document_source  \
0     0000559             GHR   
1     0000559             GHR   
2     0000559             GHR   
3     0000559             GHR   
4     0000559             GHR   

                                        document_url category  umls_cui  \
0  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
1  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
2  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
3  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   
4  https://ghr.nlm.nih.gov/condition/keratoderma-...      NaN  C0343073   

  umls_semantic_types umls_semantic_group synonyms question_id  \
0                T047           Disorders     KWWH   0000559-1   
1                T047           Disorders     KWWH   0000559-2   
2                T047           Disorders     KWWH   0000559-3   
3                T047           Disorders     KWWH   0000559-4   
4                T04

In [27]:
def create_instruction(row):
  return {
       "instruction": "Answer the following medical question accurately and clearly.",
        "input": row["question"],
        "output": row["answer"]
  }

In [28]:
processed_data=df.apply(create_instruction,axis=1).tolist()

In [29]:
print(processed_data[0])

{'instruction': 'Answer the following medical question accurately and clearly.', 'input': 'What is (are) keratoderma with woolly hair ?', 'output': 'Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appe

In [30]:
def create_training_text(row):
    return (
        "### Instruction:\n"
        "Answer the following medical question accurately and clearly.\n\n"
        "### Question:\n"
        f"{row['question']}\n\n"
        "### Answer:\n"
        f"{row['answer']}"
    )

In [31]:
df["text"]=df.apply(create_training_text,axis=1)

In [32]:
print(df["text"][0])

### Instruction:
Answer the following medical question accurately and clearly.

### Question:
What is (are) keratoderma with woolly hair ?

### Answer:
Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not a

In [33]:
print("Final dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Final dataset shape: (16359, 14)

Columns:
['document_id', 'document_source', 'document_url', 'category', 'umls_cui', 'umls_semantic_types', 'umls_semantic_group', 'synonyms', 'question_id', 'question_focus', 'question_type', 'question', 'answer', 'text']


In [34]:
df[['question','answer','text']].to_csv("processed.csv",index=False)

In [35]:
with open(
    "medical_training.jslon",
    "w",
    encoding="utf-8"
)as file:
  for item in processed_data:
    file.write(json.dumps(item,ensure_ascii=False)+"\n")


In [36]:
with open(
    "medical_training.jslon",
    "r",
    encoding="utf-8"
) as f:
    first_line = f.readline()

print(first_line)

{"instruction": "Answer the following medical question accurately and clearly.", "input": "What is (are) keratoderma with woolly hair ?", "output": "Keratoderma with woolly hair is a group of related conditions that affect the skin and hair and in many cases increase the risk of potentially life-threatening heart problems. People with these conditions have hair that is unusually coarse, dry, fine, and tightly curled. In some cases, the hair is also sparse. The woolly hair texture typically affects only scalp hair and is present from birth. Starting early in life, affected individuals also develop palmoplantar keratoderma, a condition that causes skin on the palms of the hands and the soles of the feet to become thick, scaly, and calloused.  Cardiomyopathy, which is a disease of the heart muscle, is a life-threatening health problem that can develop in people with keratoderma with woolly hair. Unlike the other features of this condition, signs and symptoms of cardiomyopathy may not appe

In [37]:
print("Final number of records:", len(df))

print("\nMissing values:")
print(df[["question", "answer", "text"]].isnull().sum())

print("\nDuplicate records:")
print(df[["question", "answer"]].duplicated().sum())

Final number of records: 16359

Missing values:
question    0
answer      0
text        0
dtype: int64

Duplicate records:
0


Base Model Selection & Loading

In [38]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")


PyTorch version: 2.13.0+cpu
CUDA available: False
GPU not available


In [39]:
!pip install -q transformers accelerate
from transformers import AutoTokenizer, AutoModelForCausalLM


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Model:", MODEL_NAME)

Model: Qwen/Qwen2.5-1.5B-Instruct


In [41]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
print("tokenizer loaded successfully")

tokenizer loaded successfully


In [42]:
text="what is diabates"
tokens=tokenizer(text)
print(tokens)


{'input_ids': [12555, 374, 1853, 370, 973], 'attention_mask': [1, 1, 1, 1, 1]}


In [43]:
print("Input_ids")
print(tokens['input_ids'])

Input_ids
[12555, 374, 1853, 370, 973]


In [ ]:
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16

)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)

print("Model loaded on:", device)

Model loaded on: cuda


In [ ]:
total_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {total_params:,}")
print(
    "Model type:",
    model.config.model_type
)

print(
    "Hidden size:",
    model.config.hidden_size
)

print(
    "Number of layers:",
    model.config.num_hidden_layers
)

print(
    "Vocabulary size:",
    model.config.vocab_size
)
if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved() / 1024**3
    )

    print("\nGPU Memory")
    print(
        f"Allocated: {allocated:.2f} GB"
    )

    print(
        f"Reserved: {reserved:.2f} GB"
    )

Total parameters: 1,543,714,304
Model type: qwen2
Hidden size: 1536
Number of layers: 28
Vocabulary size: 151936

GPU Memory
Allocated: 2.88 GB
Reserved: 3.06 GB


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is diabetes?"
    }
]

# Convert conversation into model's chat format
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("\n" + "=" * 60)
print("BASE MODEL TEST")
print("=" * 60)

# Tokenize prompt
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(device)

# Generate response
with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True
    )

# Remove original prompt tokens
input_length = inputs["input_ids"].shape[1]

generated_tokens = outputs[0][input_length:]

# Convert tokens back to text
response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("\nQuestion:")
print("What is diabetes?")

print("\nBase Model Response:")
print(response)



BASE MODEL TEST

Question:
What is diabetes?

Base Model Response:
Diabetes is a chronic disease that affects how your body uses glucose (sugar), which is a type of sugar found in food.

When you eat something containing carbohydrates, the carbohydrates break down into glucose and enter your bloodstream. Your pancreas then releases insulin, a hormone that helps cells throughout your body absorb glucose from your blood for energy or storage.

In people with diabetes, either their bodies don't produce enough insulin, or they can't use insulin effectively. This causes high levels of glucose to remain


In [ ]:
print(" Final Verification ")
print("✓ Base model:", MODEL_NAME)
print("✓ Tokenizer loaded:", tokenizer is not None)
print("✓ Model loaded:", model is not None)
print("✓ Device:", device)

if torch.cuda.is_available():
    print(
        "✓ GPU:",
        torch.cuda.get_device_name(0)
    )

 Final Verification 
✓ Base model: Qwen/Qwen2.5-1.5B-Instruct
✓ Tokenizer loaded: True
✓ Model loaded: True
✓ Device: cuda
✓ GPU: Tesla T4


QLoRA CONFIGURATION


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 54.6 MB/s eta 0:00:00


In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model
)

In [ ]:
print("check GPU")
print("CUDA Available :", torch.cuda.is_available())
if not torch.cuda.is_available():
   raise RuntimeError(
        "GPU not available. Go to Runtime > Change runtime type > T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name())
MODEL_NAME
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nTokenizer loaded successfully!")

check GPU
CUDA Available : True
GPU: Tesla T4

Tokenizer loaded successfully!


In [ ]:
#Configure 4-bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\n4-bit Quantization Configuration:")
print(bnb_config)
#Load Base Model in 4-bit
print("\nLoading model in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("4-bit model loaded successfully!")



4-bit Quantization Configuration:
BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


Loading model in 4-bit...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

4-bit model loaded successfully!


In [ ]:
# LoRA Configuration

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

print("\nLoRA configuration created:")
print(lora_config)

# Apply Lora to Model

model = get_peft_model(
    model,
    lora_config
)

print("\nLoRA adapter applied successfully!")


LoRA configuration created:
LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj', 'o_proj', 'k_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)

LoRA adapter a

In [ ]:
print("TRAINABLE PARAMETERS")


model.print_trainable_parameters()

print("Model Device",model.device)
print(torch.cuda.get_device_name(0))


if torch.cuda.is_available():

    allocated = (
        torch.cuda.memory_allocated() / 1024**3
    )

    reserved = (
        torch.cuda.memory_reserved() / 1024**3
    )

    print("\nGPU Memory:")
    print(f"Allocated: {allocated:.2f} GB")
    print(f"Reserved: {reserved:.2f} GB")


TRAINABLE PARAMETERS
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
Model Device cuda:0
Tesla T4

GPU Memory:
Allocated: 1.77 GB
Reserved: 4.11 GB
